# RF-DETR Location Tag Detection — End-to-End Pipeline

## Setup

In [ ]:
# Cell 1 — Install dependencies
%pip install -q rfdetr azure-storage-blob python-dotenv tqdm pillow


In [ ]:
# Cell 2 — Load Azure credentials
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"
connection_string = os.getenv(AZURE_CONNECTION_STRING_ENV)

if not connection_string:
    raise RuntimeError(
        f"Azure connection string not found. Set {AZURE_CONNECTION_STRING_ENV} in .env."
    )

print("Azure connection string loaded.")


In [ ]:
# Cell 3 — Configuration
from pathlib import Path

INPUT_JSON_DIR = Path("./coco_files")
DATASET_DIR = Path("./rfdetr_dataset")
OUTPUT_DIR = Path("./rfdetr_output")
PRETRAINED_WEIGHTS = Path("./weights/rf-detr-base.pth")

BBOX_FORMAT = "xywh"
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"
AREA_FIELD = "area"

TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 1

DOWNLOAD_WORKERS = 16
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

if abs(TRAIN_RATIO + VALID_RATIO + TEST_RATIO - 1.0) > 1e-8:
    raise ValueError("Train, validation and test ratios must sum to 1.0.")

print("Configuration loaded.")


In [ ]:
# Cell 4 — Imports and clients
import hashlib
import json
import random
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from urllib.parse import urlparse

from PIL import Image
from tqdm.auto import tqdm
from azure.storage.blob import BlobServiceClient
from rfdetr import RFDETRBase

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
random.seed(RANDOM_SEED)

print("Imports and Azure Blob client initialized.")


In [ ]:
# Cell 5 — Helper functions
def normalize_category(value):
    if isinstance(value, list):
        if len(value) != 1:
            raise ValueError(f"Expected one category per bbox: {value}")
        return str(value[0])
    return str(value)

def get_image_extension(image_url):
    suffix = Path(urlparse(image_url).path).suffix.lower()
    return suffix if suffix in IMAGE_EXTENSIONS else ".jpg"

def image_filename(image_url):
    digest = hashlib.md5(image_url.encode("utf-8")).hexdigest()
    return f"{digest}{get_image_extension(image_url)}"

def parse_azure_blob_url(image_url):
    parts = urlparse(image_url).path.lstrip("/").split("/", 1)
    if len(parts) != 2:
        raise ValueError(f"Could not parse Azure Blob URL: {image_url}")
    return parts[0], parts[1]

def clip_bbox_xywh(bbox, image_width, image_height):
    x, y, width, height = map(float, bbox)

    x1 = max(0.0, min(x, image_width))
    y1 = max(0.0, min(y, image_height))
    x2 = max(0.0, min(x + width, image_width))
    y2 = max(0.0, min(y + height, image_height))

    clipped_width = x2 - x1
    clipped_height = y2 - y1

    if clipped_width <= 0 or clipped_height <= 0:
        return None

    return [x1, y1, clipped_width, clipped_height]

def download_blob_image(image_url, output_path):
    container_name, blob_name = parse_azure_blob_url(image_url)

    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name,
    )

    data = blob_client.download_blob().readall()
    image = Image.open(BytesIO(data)).convert("RGB")
    image.save(output_path)

    return image.size

print("Helper functions defined.")


## Full pipeline

Progress output is intentionally limited to one overall tqdm bar per major stage. There are no nested per-image or per-bounding-box progress bars, so errors remain visible in notebook output.

### 1. Read annotation JSON files

In [ ]:
# Cell 6 — Discover JSON files
json_files = sorted(INPUT_JSON_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"No JSON files found in {INPUT_JSON_DIR.resolve()}"
    )

print(f"Found {len(json_files)} JSON files.")


In [ ]:
# Cell 7 — Read JSON files
annotation_records = []

with tqdm(total=len(json_files), desc="Reading JSON files") as progress:
    for json_file in json_files:
        with json_file.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, list):
            raise ValueError(
                f"{json_file.name} must contain a list of annotation records."
            )

        annotation_records.extend(data)
        progress.update(1)

print(f"Annotation records: {len(annotation_records)}")


### 2. Process annotations and auto-detect classes

In [ ]:
# Cell 8 — Initialize annotation structures
image_annotations = defaultdict(list)
classes = []
class_seen = set()

invalid_records = 0

print("Annotation structures initialized.")


In [ ]:
# Cell 9 — Process annotation records
with tqdm(total=len(annotation_records), desc="Processing annotations") as progress:
    for record in annotation_records:
        try:
            image_url = record[IMAGE_FIELD]
            category = normalize_category(record[CATEGORY_FIELD])
            bbox = record[BBOX_FIELD]

            if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
                raise ValueError(f"Invalid bbox: {bbox}")

            if category not in class_seen:
                class_seen.add(category)
                classes.append(category)

            image_annotations[image_url].append({
                "category": category,
                "bbox": list(map(float, bbox)),
            })

        except Exception as exc:
            invalid_records += 1
            print(f"Skipping invalid annotation: {exc}")

        progress.update(1)

if not image_annotations:
    raise RuntimeError("No valid image annotations found.")

category_to_id = {
    category: index + 1
    for index, category in enumerate(classes)
}

print(f"Unique images: {len(image_annotations)}")
print(f"Classes: {classes}")
print(f"Invalid records: {invalid_records}")


### 3. Create train / validation / test split

In [ ]:
# Cell 10 — Shuffle image URLs
image_urls = list(image_annotations.keys())
random.Random(RANDOM_SEED).shuffle(image_urls)

print(f"Images available for splitting: {len(image_urls)}")


In [ ]:
# Cell 11 — Create image-level split
total_images = len(image_urls)

train_end = int(total_images * TRAIN_RATIO)
valid_end = train_end + int(total_images * VALID_RATIO)

train_urls = image_urls[:train_end]
valid_urls = image_urls[train_end:valid_end]
test_urls = image_urls[valid_end:]

splits = {
    "train": train_urls,
    "valid": valid_urls,
    "test": test_urls,
}

print(f"Train: {len(train_urls)}")
print(f"Valid: {len(valid_urls)}")
print(f"Test : {len(test_urls)}")


### 4. Create RF-DETR dataset folders

In [ ]:
# Cell 12 — Create dataset directories
for split_name in splits:
    (DATASET_DIR / split_name / "images").mkdir(
        parents=True,
        exist_ok=True,
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset directory: {DATASET_DIR.resolve()}")


In [ ]:
# Cell 13 — Save detected classes
classes_file = DATASET_DIR / "classes.json"

with classes_file.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "classes": classes,
            "category_to_id": category_to_id,
        },
        f,
        indent=2,
    )

print(f"Saved classes to {classes_file}")


### 5. Download images from Azure

In [ ]:
# Cell 14 — Define image download worker
def download_one_image(task):
    image_url, output_path = task

    try:
        size = download_blob_image(image_url, output_path)

        return {
            "image_url": image_url,
            "output_path": output_path,
            "width": size[0],
            "height": size[1],
            "error": None,
        }

    except Exception as exc:
        return {
            "image_url": image_url,
            "output_path": output_path,
            "width": None,
            "height": None,
            "error": exc,
        }

print("Download worker defined.")


In [ ]:
# Cell 15 — Build download tasks
download_tasks = {}

for split_name, split_urls in splits.items():
    split_image_dir = DATASET_DIR / split_name / "images"

    download_tasks[split_name] = [
        (
            image_url,
            split_image_dir / image_filename(image_url),
        )
        for image_url in split_urls
    ]

print("Download tasks created.")


In [ ]:
# Cell 16 — Download train images
train_download_results = []

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = [
        executor.submit(download_one_image, task)
        for task in download_tasks["train"]
    ]

    with tqdm(total=len(futures), desc="Downloading train") as progress:
        for future in as_completed(futures):
            train_download_results.append(future.result())
            progress.update(1)

print(f"Train downloads completed: {len(train_download_results)}")


In [ ]:
# Cell 17 — Download validation images
valid_download_results = []

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = [
        executor.submit(download_one_image, task)
        for task in download_tasks["valid"]
    ]

    with tqdm(total=len(futures), desc="Downloading valid") as progress:
        for future in as_completed(futures):
            valid_download_results.append(future.result())
            progress.update(1)

print(f"Validation downloads completed: {len(valid_download_results)}")


In [ ]:
# Cell 18 — Download test images
test_download_results = []

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = [
        executor.submit(download_one_image, task)
        for task in download_tasks["test"]
    ]

    with tqdm(total=len(futures), desc="Downloading test") as progress:
        for future in as_completed(futures):
            test_download_results.append(future.result())
            progress.update(1)

print(f"Test downloads completed: {len(test_download_results)}")


In [ ]:
# Cell 19 — Check download errors
download_results = {
    "train": train_download_results,
    "valid": valid_download_results,
    "test": test_download_results,
}

failed_downloads = []

for split_name, results in download_results.items():
    for result in results:
        if result["error"] is not None:
            failed_downloads.append({
                "split": split_name,
                **result,
            })

print(f"Total failed downloads: {len(failed_downloads)}")

for failure in failed_downloads[:10]:
    print(f"[{failure['split']}] {failure['image_url']}")
    print(failure["error"])


### 6. Generate COCO annotation files

In [ ]:
# Cell 20 — Define COCO generation function
def create_coco_for_split(split_name, split_urls, split_download_results):
    image_id_map = {}
    image_sizes = {}

    for result in split_download_results:
        if result["error"] is not None:
            continue

        image_url = result["image_url"]

        image_id_map[image_url] = len(image_id_map) + 1
        image_sizes[image_url] = (
            result["width"],
            result["height"],
        )

    coco = {
        "info": {
            "description": "RF-DETR Location Tag Detection Dataset"
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [],
    }

    for category_name, category_id in category_to_id.items():
        coco["categories"].append({
            "id": category_id,
            "name": category_name,
            "supercategory": "object",
        })

    annotation_id = 1

    for image_url in split_urls:
        if image_url not in image_id_map:
            continue

        image_id = image_id_map[image_url]
        width, height = image_sizes[image_url]

        coco["images"].append({
            "id": image_id,
            "file_name": image_filename(image_url),
            "width": width,
            "height": height,
        })

        for annotation in image_annotations[image_url]:
            bbox = clip_bbox_xywh(
                annotation["bbox"],
                width,
                height,
            )

            if bbox is None:
                continue

            x, y, bbox_width, bbox_height = bbox

            coco["annotations"].append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": category_to_id[annotation["category"]],
                "bbox": bbox,
                "area": bbox_width * bbox_height,
                "iscrowd": 0,
                "segmentation": [],
            })

            annotation_id += 1

    annotation_file = (
        DATASET_DIR
        / split_name
        / "_annotations.coco.json"
    )

    with annotation_file.open("w", encoding="utf-8") as f:
        json.dump(coco, f, indent=2)

    return coco

print("COCO generation function defined.")


In [ ]:
# Cell 21 — Generate train COCO annotations
train_coco = create_coco_for_split(
    "train",
    train_urls,
    train_download_results,
)

print(
    f"Train COCO: "
    f"{len(train_coco['images'])} images, "
    f"{len(train_coco['annotations'])} annotations."
)


In [ ]:
# Cell 22 — Generate validation COCO annotations
valid_coco = create_coco_for_split(
    "valid",
    valid_urls,
    valid_download_results,
)

print(
    f"Valid COCO: "
    f"{len(valid_coco['images'])} images, "
    f"{len(valid_coco['annotations'])} annotations."
)


In [ ]:
# Cell 23 — Generate test COCO annotations
test_coco = create_coco_for_split(
    "test",
    test_urls,
    test_download_results,
)

print(
    f"Test COCO: "
    f"{len(test_coco['images'])} images, "
    f"{len(test_coco['annotations'])} annotations."
)


### 7. Dataset validation and summary

In [ ]:
# Cell 24 — Dataset summary
coco_data = {
    "train": train_coco,
    "valid": valid_coco,
    "test": test_coco,
}

for split_name, coco in coco_data.items():
    print(f"\n{split_name.upper()}")
    print(f"Images      : {len(coco['images'])}")
    print(f"Annotations : {len(coco['annotations'])}")

print("\nClasses:")
for category_name, category_id in category_to_id.items():
    print(f"{category_id}: {category_name}")


In [ ]:
# Cell 25 — Verify pretrained weights
if not PRETRAINED_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Pretrained weights not found: {PRETRAINED_WEIGHTS.resolve()}"
    )

print(f"Pretrained weights: {PRETRAINED_WEIGHTS.resolve()}")
print(
    f"Size: "
    f"{PRETRAINED_WEIGHTS.stat().st_size / (1024 ** 2):.2f} MB"
)


### 8. Initialize RF-DETR

In [ ]:
# Cell 26 — Create RF-DETR model
model = RFDETRBase(
    pretrain_weights=str(PRETRAINED_WEIGHTS)
)

print("RF-DETR model initialized.")


### 9. Train RF-DETR

In [ ]:
# Cell 27 — Train model
model.train(
    dataset_dir=str(DATASET_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    output_dir=str(OUTPUT_DIR),
)


### 10. Find trained checkpoint

In [ ]:
# Cell 28 — Find checkpoints
checkpoint_candidates = []

for pattern in ("*.pth", "*.pt"):
    checkpoint_candidates.extend(
        OUTPUT_DIR.rglob(pattern)
    )

checkpoint_candidates = sorted(
    checkpoint_candidates,
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

with tqdm(
    total=max(len(checkpoint_candidates), 1),
    desc="Searching checkpoints",
) as progress:
    for _ in checkpoint_candidates:
        progress.update(1)

if not checkpoint_candidates:
    raise FileNotFoundError(
        f"No checkpoint found under {OUTPUT_DIR.resolve()}"
    )

BEST_CHECKPOINT = checkpoint_candidates[0]

print(f"Selected checkpoint: {BEST_CHECKPOINT}")


### 11. Load trained model

In [ ]:
# Cell 29 — Load trained checkpoint
evaluation_model = RFDETRBase(
    pretrain_weights=str(BEST_CHECKPOINT)
)

print("Trained RF-DETR checkpoint loaded.")


### 12. Evaluate

In [ ]:
# Cell 30 — Evaluate model
evaluation_result = evaluation_model.evaluate(
    dataset_dir=str(DATASET_DIR)
)

print(evaluation_result)


### 13. Final summary

In [ ]:
# Cell 31 — Final pipeline summary
print("=" * 60)
print("RF-DETR LOCATION TAG DETECTION PIPELINE COMPLETE")
print("=" * 60)

print(f"Classes           : {classes}")
print(f"Train images      : {len(train_urls)}")
print(f"Validation images : {len(valid_urls)}")
print(f"Test images       : {len(test_urls)}")
print(f"Dataset directory : {DATASET_DIR.resolve()}")
print(f"Output directory  : {OUTPUT_DIR.resolve()}")
print(f"Checkpoint        : {BEST_CHECKPOINT}")
print("=" * 60)
